In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score

df = pd.read_csv("../data/telco_clean.csv")

# y = what we want to predict, X = the information we use to predict it
y = df["ChurnFlag"]
X = df.drop(columns=["customerID", "Churn", "ChurnFlag", "TenureGroup"])

# Turn text columns (Yes/No, contract types) into 0/1 columns
X = pd.get_dummies(X, drop_first=True).astype(float)
print(X.shape)

(7043, 23)


In [2]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print(X_train.shape, X_test.shape)
print("Churn rate in train:", round(y_train.mean(), 3))
print("Churn rate in test:", round(y_test.mean(), 3))

(5634, 23) (1409, 23)
Churn rate in train: 0.265
Churn rate in test: 0.265


In [3]:
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_s, y_train)

lr_probs = lr.predict_proba(X_test_s)[:, 1]
print("Logistic ROC-AUC:", round(roc_auc_score(y_test, lr_probs), 3))
print("Logistic accuracy:", round(accuracy_score(y_test, lr.predict(X_test_s)), 3))

Logistic ROC-AUC: 0.842
Logistic accuracy: 0.807


In [4]:
coefs = pd.Series(lr.coef_[0], index=X.columns)
coefs.reindex(coefs.abs().sort_values(ascending=False).index).head(10).round(2)

tenure                        -1.25
MonthlyCharges                -0.84
InternetService_Fiber optic    0.74
InternetService_No            -0.62
Contract_Two year             -0.59
TotalCharges                   0.53
Contract_One year             -0.28
StreamingMovies_Yes            0.24
StreamingTV_Yes                0.24
MultipleLines_Yes              0.21
dtype: float64

In [5]:
rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, random_state=42)
rf.fit(X_train, y_train)

rf_probs = rf.predict_proba(X_test)[:, 1]
print("Random forest ROC-AUC:", round(roc_auc_score(y_test, rf_probs), 3))

Random forest ROC-AUC: 0.844


In [6]:
imp = pd.Series(rf.feature_importances_, index=X.columns)
imp.sort_values(ascending=False).head(10).round(3)

tenure                            0.192
TotalCharges                      0.160
MonthlyCharges                    0.134
InternetService_Fiber optic       0.089
PaymentMethod_Electronic check    0.067
Contract_Two year                 0.061
Contract_One year                 0.036
InternetService_No                0.035
OnlineSecurity_Yes                0.028
PaperlessBilling_Yes              0.026
dtype: float64

In [7]:
#Risk list and dashboard

In [8]:
#Risk list and dashboard

In [9]:
import numpy as np

X_all_s = scaler.transform(X)
df["ChurnProbability"] = lr.predict_proba(X_all_s)[:, 1].round(3)

df["RiskBand"] = pd.cut(
    df["ChurnProbability"],
    bins=[-0.01, 0.3, 0.6, 1.0],
    labels=["Low", "Medium", "High"]
).astype(str)

df["RiskBand"].value_counts()

RiskBand
Low       4376
Medium    1627
High      1040
Name: count, dtype: int64

In [10]:
df.groupby("PaymentMethod")["ChurnFlag"].mean().mul(100).round(1)

PaymentMethod
Bank transfer (automatic)    16.7
Credit card (automatic)      15.2
Electronic check             45.3
Mailed check                 19.1
Name: ChurnFlag, dtype: float64

In [11]:
df.to_csv("../data/telco_scored.csv", index=False)
print("Saved!", df.shape)

Saved! (7043, 25)
